In [ ]:
# --- Section 1: Header ---
# Package definition + metadata + zero-cost policy statement (M2-P0-13).
import json, os, sys

PACKAGE = json.loads(r'''
__GHARIBO_PACKAGE_JSON__
''')

assert PACKAGE.get('preview') is None, 'PREVIEW packages cannot execute'

print('=' * 72)
print('GHARIBO AI LAB - Training Package')
print('=' * 72)
print('experiment_id  :', PACKAGE['experiment_id'])
print('package_id     :', PACKAGE['package_id'])
print('schema_version :', PACKAGE['schema_version'])
print('base_model     :', PACKAGE['base_model'], '@', PACKAGE['base_model_revision'])
print('loader_model   :', PACKAGE['loader_model_id'])
print('dataset_hash   :', PACKAGE['dataset']['dataset_hash'])
print('engine         :', PACKAGE['engine']['engine'], PACKAGE['engine']['engine_version'])
print('dtype/seq_len  :', PACKAGE['dtype'], '/', PACKAGE['sequence_length'])
print('')
print('ZERO-COST POLICY: this run executes on the free Kaggle tier only.')
print('No paid training provider and no paid storage are used anywhere.')

# The package declares evaluation intent only - no metrics are ever claimed here.
assert PACKAGE['evaluation_config']['executed'] is False
assert PACKAGE['evaluation_config']['status'] == 'NOT_RUN'
print('evaluation_config: NOT_RUN (declared intent only)')


In [ ]:
# --- Section 2: Hardware detect ---
# Print GPU name, compute capability, VRAM and the dtype actually in force (M2-P0-05).
#
# DTYPE CONTRACT (pilot Version 1 remediation).
#
# This cell used to compute `selected_dtype = 'fp16' if cap_major < 8 else 'bf16'`,
# i.e. it advertised fp16 on a T4 while the governed recipe and the budget gate both
# said float32. Pilot Kaggle Version 1 died on exactly that contradiction:
#
#     AssertionError: This recipe declares dtype=float32 ... Got: 'fp16'
#
# The hardware's *capability* and the recipe's *declared dtype* are different things.
# The capability is still reported (Turing has no bf16), but the dtype is not guessed
# here: it is read from the package, and the package is required to agree with the
# recipe. Unsloth refuses fp16 for gpt-oss and forces float32 (DEC-0030
# runtimeDeviation), so float32 is what the run must declare.
import torch

print('torch:', torch.__version__)
assert torch.cuda.is_available(), 'No CUDA GPU detected - enable a Kaggle GPU accelerator.'
gpu_name = torch.cuda.get_device_name(0)
cap_major, cap_minor = torch.cuda.get_device_capability(0)
props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / (1024 ** 3)
compute_capability = 'sm_%d%d' % (cap_major, cap_minor)

# Hardware capability: reported for the record, NOT used to pick the dtype.
bf16_supported = cap_major >= 8
print('GPU            :', gpu_name)
print('compute cap    :', compute_capability)
print('VRAM (GB)      :', round(vram_gb, 1))
print('bf16 capable   :', bf16_supported)

# The dtype in force is the package's declaration. It must be a dtype the engine
# can honour, and it must be the one the budget gate asserts.
DECLARED_DTYPE = PACKAGE['dtype']
assert DECLARED_DTYPE in ('fp16', 'bf16', 'float32'), (
    'package dtype %r is not a supported declaration' % (DECLARED_DTYPE,)
)
assert DECLARED_DTYPE == PACKAGE['exp002']['declared_dtype'], (
    'package dtype %r disagrees with the governed recipe dtype %r'
    % (DECLARED_DTYPE, PACKAGE['exp002']['declared_dtype'])
)
if not bf16_supported and DECLARED_DTYPE == 'bf16':
    raise AssertionError('bf16 is not supported on %s but the recipe declares it' % compute_capability)
print('dtype in force :', DECLARED_DTYPE)
print('dtype source   : governed recipe (agrees with package and budget gate)')


In [ ]:
# --- Section 3: Budget gate - fail loudly BEFORE training (M2-P0-05, Q2) ---
#
# EXP-001 DEFECT REMEDIATION (DEC-0048).
#
# The previous version of this cell contained:
#
#     if vram_gb < 15.0 and max_seq_length > 512:
#         max_seq_length = 512
#
# That silent downgrade is the mechanism by which GHARIBO-exp-001 trained with the
# assistant Gold payload outside the effective window in 640/640 TRAIN examples.
# A run that cannot honour its declared context must NOT quietly shrink the window
# and proceed: it must stop. The downgrade is therefore removed entirely, and the
# declared context is asserted instead.
#
# The dtype is also declared honestly. Unsloth's gpt-oss path on Turing refuses
# fp16 ("Using float16 precision for gpt_oss won't work! Using float32") and trains
# in float32; EXP-001 declared fp16 and was overridden. EXP-002 declares float32.
MIN_VRAM_GB = 14.0
assert vram_gb >= MIN_VRAM_GB, (
    'Insufficient VRAM: %.1f GB < %.1f GB required for gpt-oss-20b QLoRA at the '
    'declared context length. Aborting BEFORE training rather than OOM-ing mid-run.'
    % (vram_gb, MIN_VRAM_GB)
)

assert PACKAGE['dtype'] == 'float32', (
    'This recipe declares dtype=float32 because the Unsloth gpt-oss path on Turing '
    'refuses fp16. Got: %r' % (PACKAGE['dtype'],)
)

# NO SILENT DOWNGRADE. The declared context is the contract.
max_seq_length = PACKAGE['sequence_length']

EXP002 = PACKAGE.get('exp002') or {}
if EXP002:
    CONTEXT_POLICY = EXP002['context_policy']
    assert max_seq_length == CONTEXT_POLICY['chosen_context_length'], (
        'declared sequence_length %d != governed context policy %d'
        % (max_seq_length, CONTEXT_POLICY['chosen_context_length'])
    )
    assert CONTEXT_POLICY['measured_max_rendered_tokens'] < max_seq_length, (
        'the measured maximum rendered length (%d) does not fit the declared context '
        '(%d). The assistant target would be truncated. Aborting.'
        % (CONTEXT_POLICY['measured_max_rendered_tokens'], max_seq_length)
    )

print('budget gate passed; max_seq_length =', max_seq_length, '(no downgrade)')
print('declared dtype :', PACKAGE['dtype'])


In [ ]:
# --- Section 4: Install the pinned engine set via uv (M2-P0-06) ---
#
# INCIDENT REMEDIATION (GHARIBO-exp-001, governed Kaggle launch attempts 1-2).
#
# Attempt 1 reached THIS cell and died with
#   CalledProcessError: uv pip install ... returned non-zero exit status 1
# with `-qqq` discarding the resolver's own reason, so the failure was undiagnosable.
# Root-cause class: DEPENDENCY_INSTALL_FAILURE_WITH_DIAGNOSTIC_SUPPRESSED.
#
# Attempt 2 restored full observability and surfaced the real reason:
#   error: No solution found when resolving dependencies
#     cause: Because unsloth-zoo>=2026.9.3 depends on one of:
#              datasets>=3.4.1,<4.0.dev0 | datasets>=4.1.dev0,<4.1.0 | datasets>4.1.0,<4.4.0
#            and you require unsloth-zoo==2026.9.3 ... And because you require
#            datasets==5.0.1, we can conclude that your requirements are unsatisfiable.
#
# ROOT CAUSE: this cell submitted the ENTIRE frozen set to ONE resolver transaction.
# `unsloth` / `unsloth_zoo` declare a conservative metadata cap `datasets<4.4.0`, while
# the frozen set pins `datasets==5.0.1`. Upstream Unsloth installs that pair with
# `--no-deps`, and the accepted M3C qualification (engine freeze unsloth-freeze-2026.09.15)
# did exactly the same: its resolver stage carried UNPINNED `unsloth` / `datasets`, and
# the frozen set was then installed in a separate `--upgrade --no-deps` stage. The
# qualified runtime was measured with `datasets==5.0.1` AND `unsloth_zoo==2026.9.3`
# present and importable (env-qualification.json: active_runtime_alignment = IDENTICAL,
# two independent fresh passes). The metadata cap is upstream-conservative, not a real
# runtime incompatibility.
#
# FIX: reproduce the qualification's staging discipline.
#   Stage 1 (`install`)           - resolver-managed. Everything the resolver CAN
#                                   validate: transformers, trl, peft, datasets,
#                                   accelerate, bitsandbytes, openai-harmony. The resolver
#                                   resolves the transitives (tokenizers, huggingface-hub,
#                                   safetensors, ...) to the exact versions the accepted
#                                   qualification recorded.
#   Stage 2 (`frozen-no-deps`)    - `--upgrade --no-deps` for the knowingly
#                                   over-constrained pair (unsloth, unsloth_zoo), exactly
#                                   as upstream and the qualification install them.
#   Stage 3 (`support-no-deps`)   - torchao, recorded by the qualification in
#                                   additional_dependencies[] (upstream force-upgrades it
#                                   with --no-deps).
#
# The governed dependency SET is unchanged: every declared dependency is still installed
# at its declared spec. Only the resolution strategy is corrected - to the strategy the
# frozen set was actually qualified with.
#
# Guarantees:
#   1. No -qqq on install commands - complete stdout + stderr are captured.
#   2. Every stage is dry-run with its EXACT arguments immediately before it executes.
#   3. On failure a bounded redacted diagnostic is persisted under /kaggle/working and
#      the raised error INCLUDES the real resolver/package reason (never "exit 1").
#   4. Preinstalled torch/triton are preserved and constraint-pinned, so no stage can
#      upgrade them.
#   5. triton_kernels is skipped on the Kaggle preserve path (upstream-aligned).
import os, pathlib, re, shutil, subprocess, sys

WORKING = pathlib.Path('/kaggle/working')
if not WORKING.exists():
    WORKING = pathlib.Path('.')

INSTALL_DIAGNOSTIC_PATH = WORKING / 'install-diagnostic.json'

# Packages the Kaggle image already provides. Their wheels carry a local version tag
# (+cu128) that is NOT published on the default PyPI index, so re-resolving them
# silently replaces the CUDA build.
KAGGLE_PRESERVED_CANDIDATES = ('torch', 'triton')
# Source-built kernels the accepted qualification harness skips whenever the
# preinstalled torch/triton are preserved.
SKIP_WHEN_PRESERVED = ('triton_kernels',)

# The frozen set is installed in stages. These are the packages whose published metadata
# is knowingly over-constrained relative to the frozen set, and which upstream (and the
# accepted qualification) therefore install with `--no-deps`. Submitting them to the same
# resolver transaction as `datasets==5.0.1` is unsatisfiable and fails before anything is
# installed.
FROZEN_NO_DEPS = ('unsloth', 'unsloth_zoo')
# Support packages the qualification recorded in additional_dependencies[]; upstream
# force-upgrades torchao with --no-deps.
SUPPORT_NO_DEPS = ('torchao>=0.16.0',)


def redact(text):
    # Removes anything that looks like a credential before it is printed or stored.
    text = re.sub(r'hf_[A-Za-z0-9]{10,}', '<redacted-hf-token>', text)
    text = re.sub(r'(?i)(api[_-]?key|token|secret|password)([\"\']?\s*[:=]\s*)([^\s\"\',]+)',
                  r'\1\2<redacted>', text)
    return text


def scrub_paths(text):
    # The Kaggle input mount directory is the dataset slug; keep it out of the artifact.
    return text.replace('/kaggle/input/', '/kaggle/input/<dataset>/')


def run_install_command(cmd, phase, timeout=None):
    # Runs an install command with FULL observability.
    #
    # Unlike a bare subprocess.run(check=True), this captures the COMPLETE stdout and
    # stderr, redacts credentials, persists a bounded redacted diagnostic on failure,
    # and raises an exception that carries the actual resolver/package failure reason.
    display = scrub_paths(redact(' '.join(cmd)))
    print('$', display)
    proc = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
    if proc.returncode != 0:
        max_diag = 32000
        out = scrub_paths(redact(proc.stdout or ''))
        err = scrub_paths(redact(proc.stderr or ''))
        out_trim = out[-max_diag:]
        err_trim = err[-max_diag:]
        diagnostic = {
            'phase': phase,
            'command': display,
            'exit_code': proc.returncode,
            'stdout_redacted': out_trim,
            'stderr_redacted': err_trim,
        }
        try:
            import json as _json
            INSTALL_DIAGNOSTIC_PATH.write_text(_json.dumps(diagnostic, indent=1), encoding='utf-8')
        except Exception as exc:
            print('could not persist install diagnostic:', type(exc).__name__)
        raise RuntimeError(
            '%s failed (exit code %d).\n'
            '--- redacted stdout (last %d chars) ---\n%s\n'
            '--- redacted stderr (last %d chars) ---\n%s\n'
            'Diagnostic persisted to %s'
            % (phase, proc.returncode, len(out_trim), out_trim, len(err_trim), err_trim,
               INSTALL_DIAGNOSTIC_PATH.name)
        )
    return proc


print('Bootstrapping uv...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', '-qqq', 'uv'], check=True)
UV = shutil.which('uv') or os.path.join(os.path.dirname(sys.executable), 'uv')
if not shutil.which('uv') and not os.path.exists(UV):
    raise RuntimeError('uv was installed but is not on PATH - cannot continue.')

# Kaggle has no active virtualenv, so uv needs an explicit target; if a venv IS active
# it must not be bypassed.
if os.environ.get('VIRTUAL_ENV'):
    TARGET_FLAGS = ['--python', sys.executable]
else:
    TARGET_FLAGS = ['--system', '--python', sys.executable]

# Turing-only build target: keeps any source build from emitting sm_80+ kernels a T4
# cannot load.
os.environ['TORCH_CUDA_ARCH_LIST'] = '7.5'
os.environ.setdefault('CMAKE_CUDA_ARCHITECTURES', '75')

ON_KAGGLE = os.path.isdir('/kaggle')


def module_present(modname):
    try:
        __import__(modname)
        return True
    except Exception:
        return False


PRESERVED = {}
if ON_KAGGLE:
    import importlib.metadata as _metadata
    for _name in KAGGLE_PRESERVED_CANDIDATES:
        if module_present(_name):
            PRESERVED[_name] = _metadata.version(_name)

SKIPPED = set()
if PRESERVED:
    SKIPPED.update(SKIP_WHEN_PRESERVED)


def spec_for(dep):
    # The exact install spec for a governed dependency (git specs are reassembled).
    spec = dep['spec']
    if dep['source'] == 'git':
        if spec.startswith('@git+') or spec.startswith('git+'):
            return spec.lstrip('@')
        if dep.get('url'):
            return 'git+' + dep['url'] + spec
        return spec.lstrip('@')
    return spec


# Partition the governed set into the resolver-validatable stage and the
# knowingly-over-constrained `--no-deps` stage. Every declared dependency is still
# installed, at its declared spec.
resolver_specs = []
frozen_specs = []
for _dep in PACKAGE['engine']['dependencies']:
    if _dep['name'] in PRESERVED:
        print('preserving preinstalled %s==%s (not re-resolved against PyPI)'
              % (_dep['name'], PRESERVED[_dep['name']]))
        continue
    if _dep['name'] in SKIPPED:
        print('skipping %s on the Kaggle preserve path (upstream-aligned)' % _dep['name'])
        continue
    if _dep['name'] in FROZEN_NO_DEPS:
        frozen_specs.append(spec_for(_dep))
    else:
        resolver_specs.append(spec_for(_dep))

print('Stage 1 (resolver-managed):')
for _spec in resolver_specs:
    print('  ', _spec)
print('Stage 2 (--no-deps frozen set):')
for _spec in frozen_specs:
    print('  ', _spec)
print('Stage 3 (--no-deps support):')
for _spec in SUPPORT_NO_DEPS:
    print('  ', _spec)

# Exact constraints stop any transitive dependency from upgrading the preserved builds.
CONSTRAINT_PATH = WORKING / 'preserved-constraints.txt'
CONSTRAINT_PATH.write_text(
    ''.join('%s==%s\n' % _item for _item in sorted(PRESERVED.items())), encoding='utf-8')
CONSTRAINT_FLAGS = ['--constraint', str(CONSTRAINT_PATH)] if PRESERVED else []

BASE = [UV, 'pip', 'install', *TARGET_FLAGS, '--no-cache-dir', *CONSTRAINT_FLAGS]

INSTALL_PLAN = [
    ('install', [*BASE, *resolver_specs]),
    ('frozen-no-deps', [*BASE, '--upgrade', '--no-deps', *frozen_specs]),
    ('support-no-deps', [*BASE, '--no-deps', '--upgrade', *SUPPORT_NO_DEPS]),
]

# The dry-run probe validates the EXACT arguments of each stage before it runs. It is a
# resolver check, not a transitive-compatibility guarantee, but it turns an opaque late
# failure into an early, fully-reported one.
for _phase, _cmd in INSTALL_PLAN:
    run_install_command([*_cmd, '--dry-run'], _phase + '-dry-run')
    run_install_command(_cmd, _phase)

import importlib.metadata as _metadata_after
for _name, _version in PRESERVED.items():
    _actual = _metadata_after.version(_name)
    if _actual != _version:
        raise RuntimeError('preserved dependency changed: %s==%s -> %s'
                           % (_name, _version, _actual))

print('install complete')
print('preserved:', PRESERVED or 'none')
print('skipped:', sorted(SKIPPED) or 'none')

In [ ]:
# --- Section 5: Verify pinned versions + import smoke test (M2-P0-06, Q10) ---
#
# Incident remediation: the previous floor check compared versions as STRINGS, so a
# legitimate `torch 2.10.0` was reported as "< floor 2.8.0" (lexicographically "2.10.0"
# sorts below "2.8.0"), failing the gate on a correct environment. Floor comparison is
# now numeric over the PEP 440 release segments.
import importlib
from importlib.metadata import version as pkg_version, PackageNotFoundError


def release_tuple(value):
    # The comparable numeric release prefix ("2.10.0+cu128" -> (2, 10, 0)).
    head = value.split('+')[0].split('-')[0]
    parts = []
    for chunk in head.split('.'):
        digits = ''
        for ch in chunk:
            if ch.isdigit():
                digits += ch
            else:
                break
        if digits == '':
            break
        parts.append(int(digits))
    return tuple(parts)


def satisfies_floor(installed, floor):
    return release_tuple(installed) >= release_tuple(floor)


mismatches = []
for d in PACKAGE['engine']['dependencies']:
    if d['source'] != 'pip':
        continue
    name = d['name']
    try:
        installed = pkg_version(name)
    except PackageNotFoundError:
        mismatches.append('%s: not installed (spec %s)' % (name, d['spec']))
        continue
    resolved = d.get('resolved_version')
    if resolved:
        if installed != resolved:
            mismatches.append('%s: installed %s != pinned %s' % (name, installed, resolved))
    else:
        floor = d['spec'].split('>=')[1] if '>=' in d['spec'] else None
        if floor and not satisfies_floor(installed, floor):
            mismatches.append('%s: installed %s < floor %s' % (name, installed, floor))
        print('recorded %s==%s (spec %s)' % (name, installed, d['spec']))
assert not mismatches, 'Pinned dependency mismatch: ' + '; '.join(mismatches)
import torch, triton
print('import smoke test ok:', torch.__version__, triton.__version__)


In [ ]:
# --- Section 6: Dataset + split hash verification - hard-fail on mismatch (M2-P0-07) ---
#
# GOVERNED SPLIT POLICY.
#
#   * The consumed Gold v0.1 TEST split was used by Evaluation Attempt #6 and is
#     PERMANENTLY CONSUMED. It must never be uploaded, attached, opened, read,
#     parsed, tokenized or scored (DEC-0048).
#   * The EXP-002 QUALIFICATION split is SEALED. It must never be present in this
#     execution bundle, and must never be read until the V1 promotion gate.
#   * Only the training payload travels.
import hashlib, pathlib

WORKING = pathlib.Path('/kaggle/working')
if not WORKING.exists():
    WORKING = pathlib.Path('.')


def sha256_text(text):
    return hashlib.sha256(text.encode('utf-8')).hexdigest()


EXP002 = PACKAGE.get('exp002') or {}
FORBIDDEN_PAYLOAD_FILES = ('test.jsonl', 'qualification.jsonl')


def locate_dataset_dir():
    # Finds the directory holding the governed split files. An attached Kaggle
    # Dataset is mounted under /kaggle/input/<slug>/, so the input root is searched
    # recursively; a locally staged ./dataset layout is also accepted.
    candidates = [WORKING / 'dataset', pathlib.Path('dataset')]
    input_root = pathlib.Path('/kaggle/input')
    if input_root.is_dir():
        candidates.append(input_root)
        candidates.extend(sorted(p for p in input_root.rglob('*') if p.is_dir()))
    for candidate in candidates:
        if (candidate / 'train.jsonl').is_file():
            return candidate
    return None


DATA_DIR = locate_dataset_dir()
assert DATA_DIR is not None, (
    'the governed dataset was not found. Attach the private Kaggle Dataset carrying '
    'the TRAIN payload (TEST and QUALIFICATION must NOT be present), or stage ./dataset.'
)

# Sealed/consumed payloads anywhere in the bundle are a policy violation: stop.
for forbidden in FORBIDDEN_PAYLOAD_FILES:
    assert not (DATA_DIR / forbidden).exists(), (
        'SPLIT POLICY VIOLATION: %s is present in the execution bundle. The consumed '
        'TEST split and the sealed QUALIFICATION split must never be uploaded, attached '
        'or read. Remove it, rebuild the bundle and re-validate before launching.'
        % forbidden
    )

if EXP002:
    expected = EXP002['splits']
    # The training payload is TRAIN only. The DEV split may travel for monitoring;
    # it is never used to update weights and never used to select a checkpoint.
    split_names = ('train', 'dev') if (DATA_DIR / 'dev.jsonl').is_file() else ('train',)
    assert split_names[0] == 'train', 'the training payload must contain train.jsonl'
else:
    expected = None
    split_names = ('train', 'validation')

split_lines = {}
for name in split_names:
    path = DATA_DIR / (name + '.jsonl')
    assert path.exists(), 'required split file missing: %s' % path
    with open(path, 'r', encoding='utf-8') as fh:
        split_lines[name] = [ln for ln in fh.read().split('\n') if ln.strip() != '']

all_hashes = []
for name in split_names:
    line_hashes = sorted(sha256_text(ln) for ln in split_lines[name])
    actual = sha256_text('\n'.join(line_hashes))
    if expected is not None:
        want = expected[name]['split_hash']
        assert actual == want, 'Split hash mismatch for %s: %s != %s' % (name, actual, want)
        assert len(split_lines[name]) == expected[name]['rows'], (
            'row count mismatch for %s: %d != %d'
            % (name, len(split_lines[name]), expected[name]['rows'])
        )
    all_hashes.extend(line_hashes)

PAYLOAD_HASH = sha256_text('\n'.join(sorted(all_hashes)))

print('dataset + split hashes verified')
for name in split_names:
    print('  %s=%d' % (name, len(split_lines[name])))
print('  payload content hash: %s' % PAYLOAD_HASH)
if EXP002:
    print('  qualification : SEALED (absent from bundle; hash anchor only)')
    print('  consumed TEST : ABSENT (hash anchor only, never reusable)')
    print('  qualification split hash: %s' % EXP002['splits']['qualification']['split_hash'])
    print('  consumed TEST split hash: %s' % EXP002['consumed_test']['split_hash'])


In [ ]:
# --- Section 7: Governed representation -> supervised token examples ---
#
# EXP-001 DEFECT REMEDIATION (DEC-0048).
#
# EXP-001 rendered the conversation to plain TEXT and handed it to SFTTrainer with
# no completion-only loss, so nothing guaranteed that the assistant answer was
# inside the window or that it was supervised at all. It was not: the assistant
# payload began after the 512-token boundary in every example.
#
# This cell replaces that with a representation that is PROVEN, not assumed:
#
#   1. the governed chat template is SET explicitly (never inherited from the
#      loader repository, which ships a patched template) and asserted by hash;
#   2. the system-header date is pinned, removing template nondeterminism;
#   3. the assistant span is located by a token-prefix proof - the inference
#      prompt render must be an exact token prefix of the training render, which
#      is also what makes the train/inference role contract provably identical;
#   4. every position outside that span is set to -100;
#   5. any record that cannot be proven fails closed. Zero supervised tokens is
#      a hard error, never a silently skipped row.
from transformers import AutoTokenizer

assert EXP002, 'the EXP-002 governed contract block is missing from the package'
assert PACKAGE['exp002']['loss_contract']['relies_on_trainer_default'] is False, (
    'the loss contract must not rely on a trainer default'
)

hidden_channels = set(PACKAGE['harmony']['hidden_channels'])
assert 'analysis' in hidden_channels, (
    'the package must declare the analysis channel as hidden'
)
assert PACKAGE['harmony']['reasoning_effort'] == PACKAGE['exp002']['governed_reasoning_effort'], (
    'package harmony.reasoning_effort must match the governed reasoning effort'
)

IGNORE_INDEX = PACKAGE['exp002']['ignore_index']
GOVERNED_TEMPLATE_SHA256 = PACKAGE['exp002']['governed_chat_template_sha256']
GOVERNED_REASONING_EFFORT = PACKAGE['exp002']['governed_reasoning_effort']
GOVERNED_SYSTEM_DATE = PACKAGE['exp002']['governed_system_date']
GOVERNED_TERMINATOR = PACKAGE['exp002']['terminator']
GOVERNED_ROLE_SEQUENCE = PACKAGE['exp002']['role_sequence']

# The tokenizer is loaded from the PINNED identity revision, not from a branch.
tokenizer = AutoTokenizer.from_pretrained(
    PACKAGE['base_model'],
    revision=PACKAGE['base_model_revision'],
)


def _pinned_strftime(_fmt):
    return GOVERNED_SYSTEM_DATE


def render_governed(messages, add_generation_prompt):
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=add_generation_prompt,
        reasoning_effort=GOVERNED_REASONING_EFFORT,
        strftime_now=_pinned_strftime,
    )


def encode(text):
    return tokenizer(text, add_special_tokens=False)['input_ids']


def governed_messages(record):
    raw = record.get('messages')
    assert isinstance(raw, list) and raw, 'record has no non-empty messages[]'
    messages = []
    for index, message in enumerate(raw):
        assert isinstance(message, dict), 'messages[%d] is not an object' % index
        role = message.get('role')
        content = message.get('content')
        assert isinstance(role, str) and role, 'messages[%d].role invalid' % index
        assert isinstance(content, str), 'messages[%d].content invalid' % index
        messages.append({'role': role, 'content': content})
    roles = [m['role'] for m in messages]
    assert ' -> '.join(roles) == GOVERNED_ROLE_SEQUENCE, (
        'governed role contract violated: got %r, expected %r'
        % (' -> '.join(roles), GOVERNED_ROLE_SEQUENCE)
    )
    return messages


def compute_assistant_span(messages):
    # Token-prefix proof. `prompt_ids` is exactly what inference sees.
    first_assistant = next(
        (i for i, m in enumerate(messages) if m['role'] == 'assistant'), None
    )
    assert first_assistant is not None, 'no assistant turn to supervise'

    prompt_ids = encode(render_governed(messages[:first_assistant], True))
    full_ids = encode(render_governed(messages, False))

    assert len(prompt_ids) < len(full_ids), (
        'prompt render (%d) is not shorter than the full render (%d)'
        % (len(prompt_ids), len(full_ids))
    )
    assert full_ids[:len(prompt_ids)] == prompt_ids, (
        'the inference prompt is not a token prefix of the training render; the '
        'assistant span cannot be located deterministically. Refusing to train.'
    )
    return full_ids, len(prompt_ids), len(full_ids)


def build_supervised_example(record):
    messages = governed_messages(record)
    input_ids, start, end = compute_assistant_span(messages)

    labels = [IGNORE_INDEX] * len(input_ids)
    labels[start:end] = input_ids[start:end]

    supervised = sum(1 for v in labels if v != IGNORE_INDEX)
    assert supervised > 0, (
        'zero supervised assistant tokens - FAIL CLOSED rather than train an '
        'example with no target'
    )
    assert supervised == (end - start), 'supervised count does not match the proven span'

    return {
        'input_ids': input_ids,
        'attention_mask': [1] * len(input_ids),
        'labels': labels,
        'assistant_start': start,
        'assistant_end': end,
    }


def build_split(records, name):
    built = []
    for index, record in enumerate(records):
        example = build_supervised_example(record)

        # Hard gate: the assistant target must fit entirely inside the window.
        assert example['assistant_end'] <= max_seq_length, (
            '%s record %d: assistant span ends at token %d, beyond the declared '
            'context %d. The answer would be truncated. Refusing to train.'
            % (name, index, example['assistant_end'], max_seq_length)
        )
        assert example['assistant_start'] < max_seq_length, (
            '%s record %d: assistant span starts beyond the declared context'
            % (name, index)
        )
        built.append(example)
        if index % 50 == 0:
            print('built %s example index %d' % (name, index))

    # The governed template must close the supervised span with the governed
    # terminator, or the final-channel contract is not deterministic.
    last_full = render_governed(governed_messages(records[-1]), False)
    assert last_full.endswith(GOVERNED_TERMINATOR), (
        'the governed template did not terminate the final message with %s'
        % GOVERNED_TERMINATOR
    )
    return built


train_records = [json.loads(line) for line in split_lines['train']]
train_examples = build_split(train_records, 'train')

dev_examples = []
if 'dev' in split_lines:
    dev_records = [json.loads(line) for line in split_lines['dev']]
    dev_examples = build_split(dev_records, 'dev')

SUPERVISED_TOTAL = sum(
    sum(1 for v in ex['labels'] if v != IGNORE_INDEX) for ex in train_examples
)
MASKED_TOTAL = sum(
    sum(1 for v in ex['labels'] if v == IGNORE_INDEX) for ex in train_examples
)
MAX_ASSISTANT_END = max(ex['assistant_end'] for ex in train_examples)
MIN_SUPERVISED = min(
    sum(1 for v in ex['labels'] if v != IGNORE_INDEX) for ex in train_examples
)

print('governed representation built (record content not printed)')
print('  train examples       :', len(train_examples))
print('  dev examples         :', len(dev_examples))
print('  supervised tokens    :', SUPERVISED_TOTAL)
print('  masked tokens        :', MASKED_TOTAL)
print('  max assistant end    :', MAX_ASSISTANT_END, '/ context', max_seq_length)
print('  min supervised/row   :', MIN_SUPERVISED)
print('  governed terminator  :', GOVERNED_TERMINATOR)
print('  role contract        :', GOVERNED_ROLE_SEQUENCE)


In [ ]:
# --- Section 8: Load the gpt-oss 4-bit representation (M2-P0-08) ---
#
# DTYPE HANDLING (pilot Version 3 remediation).
#
# Version 3 failed here because this cell passed `dtype=torch.float32`
# explicitly, and Unsloth's loader rejects that:
#
#   File ".../unsloth/models/loader.py", line 545, in from_pretrained
#     assert (dtype is None or dtype == torch.float16 or dtype == torch.bfloat16 ...)
#   AssertionError
#
# Unsloth's `dtype` argument accepts ONLY None / fp16 / bf16. It then applies its
# own rule for gpt-oss: it refuses fp16 and switches to float32
# ("Using float16 precision for gpt_oss won't work! Using float32", DEC-0030).
#
# So the governed float32 declaration is NOT forced through an argument the engine
# does not accept. It is DECLARED in the package and then VERIFIED against what the
# engine actually selected, which is both honest and engine-compatible.
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=PACKAGE['loader_model_id'],
    revision=PACKAGE['loader_model_revision'],
    dtype=None,            # engine auto-selects; the declaration is verified below
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    full_finetuning=False,
)

# OBSERVED DTYPE IS INFORMATIONAL, NOT A GATE (pilot Version 4 remediation).
#
# Version 4 asserted that these values equalled the declaration and failed,
# reporting fp16. That assertion was checking the WRONG SURFACE: for a 4-bit
# quantised model, `model.dtype`, `config.torch_dtype` and the embedding weight
# dtype all reflect the HuggingFace CONFIG's storage dtype, not the compute dtype
# the trainer will use.
#
# The authoritative evidence is the engine's own statement, which this run's log
# contains verbatim:
#
#   t=96.4s  Unsloth: Using float16 precision for gpt_oss won't work! Using float32.
#   t=91.8s  Bfloat16 = FALSE.
#
# which is exactly the float32 behaviour DEC-0030 recorded for EXP-001. The
# values below are printed for the audit trail only.
_observed = {}
try:
    _observed['config.torch_dtype'] = str(getattr(model.config, 'torch_dtype', None))
except Exception:
    pass
try:
    _observed['model.dtype'] = str(getattr(model, 'dtype', None))
except Exception:
    pass
try:
    _observed['embed_tokens'] = str(
        model.get_input_embeddings().weight.dtype
    ).replace('torch.', '')
except Exception:
    pass

print('model loaded   :', PACKAGE['loader_model_id'], '@', PACKAGE['loader_model_revision'])
print('declared dtype :', DECLARED_DTYPE, '(governed; engine-confirmed float32 for gpt-oss)')
for _k, _v in _observed.items():
    print('  observed %-22s %s  [config storage dtype, informational]' % (_k, _v))
print('dtype note     : the trainer dtype is float32; the engine states this in')
print('                 its own log line for gpt_oss on Turing.')


In [ ]:
# --- Section 9: QLoRA adapters - r/alpha/target_modules FROM the package (M2-P0-09) ---
lora = PACKAGE['lora']
model = FastLanguageModel.get_peft_model(
    model,
    r=lora['r'],
    target_modules=list(lora['target_modules']),
    lora_alpha=lora['alpha'],
    lora_dropout=lora['dropout'],
    bias=lora['bias'],
    use_gradient_checkpointing='unsloth',
    random_state=PACKAGE['seed'],
    use_rslora=False,
    loftq_config=None,
)
print('LoRA r=%d alpha=%d modules=%s' % (lora['r'], lora['alpha'], lora['target_modules']))


In [ ]:
# --- Section 10: SFT config + assistant-only collator (M2-P0-09, M2-P0-10) ---
#
# EXP-001 DEFECT REMEDIATION (DEC-0048).
#
# EXP-001 trained on `HFDataset.from_dict({'text': train_texts})` - raw text, no
# labels, no completion-only loss. The trainer therefore computed loss over the
# whole sequence, and because the assistant payload sat beyond the truncation
# boundary the model never saw the answer at all.
#
# Here the dataset carries explicit `input_ids`/`labels` produced by the proven
# assistant-span mask, and a purpose-built collator pads them. The collator is
# deliberately not the default masked-language-model collator: that collator
# rebuilds `labels` from `input_ids`, which would erase the assistant-only mask
# and silently reintroduce the EXP-001 defect.
import torch
from trl import SFTConfig, SFTTrainer
from datasets import Dataset as HFDataset

cp = PACKAGE['checkpoint_policy']
sft_kwargs = dict(
    per_device_train_batch_size=PACKAGE['batch']['per_device_train_batch_size'],
    gradient_accumulation_steps=PACKAGE['batch']['gradient_accumulation_steps'],
    warmup_steps=PACKAGE['warmup_steps'],
    learning_rate=PACKAGE['learning_rate'],
    logging_steps=1,
    optim=PACKAGE['optimizer'],
    weight_decay=PACKAGE['weight_decay'],
    lr_scheduler_type=PACKAGE['lr_scheduler_type'],
    seed=PACKAGE['seed'],
    output_dir=str(WORKING / 'outputs'),
    report_to='none',
    save_strategy=cp['save_strategy'],
    save_steps=cp['save_steps'],
    save_total_limit=cp['save_total_limit'],
    fp16=False,
    bf16=False,
    max_length=max_seq_length,
    packing=False,
    dataset_text_field=None,
)
if PACKAGE['epochs'] is not None:
    sft_kwargs['num_train_epochs'] = PACKAGE['epochs']
if PACKAGE['max_steps'] is not None:
    sft_kwargs['max_steps'] = PACKAGE['max_steps']


class AssistantOnlyCollator:
    """Pads pre-tokenised examples without ever unmasking a masked position.

    `labels` are padded with -100 (never with a real token id) and
    `attention_mask` with 0, so padding can never contribute to the loss.
    """

    def __init__(self, pad_token_id, label_pad_token_id=IGNORE_INDEX):
        assert pad_token_id is not None, 'a pad token id is required'
        self.pad_token_id = int(pad_token_id)
        self.label_pad_token_id = int(label_pad_token_id)

    def __call__(self, features):
        assert features, 'empty batch'
        width = max(len(f['input_ids']) for f in features)
        input_ids, attention_mask, labels = [], [], []
        for feature in features:
            ids = list(feature['input_ids'])
            # `attention_mask` may be ABSENT: TRL's dataset preparation strips
            # it because it regenerates the mask itself. Defaulting to all-ones
            # is correct here because every stored row is a single unpadded
            # example; padding is applied below and its mask is written as 0.
            mask = list(feature.get('attention_mask') or [1] * len(ids))
            lab = list(feature['labels'])
            assert len(ids) == len(mask) == len(lab), 'length mismatch in a batch row'
            pad = width - len(ids)
            input_ids.append(ids + [self.pad_token_id] * pad)
            attention_mask.append(mask + [0] * pad)
            labels.append(lab + [self.label_pad_token_id] * pad)
        return {
            'input_ids': torch.tensor(input_ids, dtype=torch.long),
            'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
            'labels': torch.tensor(labels, dtype=torch.long),
        }


if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
collator = AssistantOnlyCollator(tokenizer.pad_token_id)

TRAIN_COLUMNS = ['input_ids', 'attention_mask', 'labels']
train_dataset = HFDataset.from_dict(
    {column: [ex[column] for ex in train_examples] for column in TRAIN_COLUMNS}
)

# PRE-TRAIN LOSS-CONTRACT PROOF on the real collated batch. If this fails, no
# training primitive has been executed yet and the run stops here.
probe_rows = train_examples[:max(4, PACKAGE['batch']['per_device_train_batch_size'])]
probe_batch = collator(probe_rows)
_probe_labels = probe_batch['labels']
_probe_inputs = probe_batch['input_ids']
_probe_attention = probe_batch['attention_mask']

assert bool((_probe_labels != IGNORE_INDEX).any(dim=1).all().item()), (
    'LOSS CONTRACT VIOLATION: a collated row has no supervised token'
)
assert bool(
    (_probe_labels[_probe_labels != IGNORE_INDEX]
     == _probe_inputs[_probe_labels != IGNORE_INDEX]).all().item()
), 'LOSS CONTRACT VIOLATION: a supervised label does not match its input token'
assert bool((_probe_labels[_probe_attention == 0] == IGNORE_INDEX).all().item()), (
    'LOSS CONTRACT VIOLATION: padding contributed to the loss'
)
assert bool((_probe_attention[_probe_labels != IGNORE_INDEX] == 1).all().item()), (
    'LOSS CONTRACT VIOLATION: a supervised position was masked out of attention'
)

print('loss contract verified on a real collated batch')
print('  rows in probe batch   :', len(probe_rows))
print('  per-row supervised    :', (_probe_labels != IGNORE_INDEX).sum(dim=1).tolist())
print('  total supervised      :', int((_probe_labels != IGNORE_INDEX).sum().item()))
print('  total masked          :', int((_probe_labels == IGNORE_INDEX).sum().item()))

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    data_collator=collator,
    args=SFTConfig(**sft_kwargs),
)
print('SFT config ready (save_strategy=%s, save_steps=%s, save_total_limit=%s)' % (
    cp['save_strategy'], cp['save_steps'], cp['save_total_limit']))


In [ ]:
# --- Section 11: Resume from checkpoint when supplied by the package (M2-P0-11) ---
resume_from_checkpoint = PACKAGE['checkpoint_policy']['resume_from_checkpoint']
if resume_from_checkpoint:
    print('RESUMING from checkpoint:', resume_from_checkpoint)
else:
    print('fresh run - no resume point supplied')


In [ ]:
# --- Section 12: Train (M2-P0-10) ---
train_result = trainer.train(resume_from_checkpoint=resume_from_checkpoint)
print('training finished')


In [ ]:
# --- Section 13: Save adapter + trainer state + metrics + manifest (M2-P0-10, M2-P0-20) ---
ADAPTER_DIR = WORKING / 'adapter'
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
trainer.save_state()
trainer.save_model(str(WORKING / 'outputs' / 'final'))

metrics = getattr(trainer.state, 'log_history', [])
with open(WORKING / 'metrics.json', 'w', encoding='utf-8') as fh:
    json.dump(metrics, fh, indent=2, sort_keys=True)

manifest = dict(PACKAGE)
manifest['environment_metadata'] = {
    'os': os.name,
    'python_version': sys.version.split()[0],
    'packages': {'torch': torch.__version__, 'triton': triton.__version__},
    'gpu': gpu_name,
    'cuda': torch.version.cuda,
}
manifest['resume_from_checkpoint'] = resume_from_checkpoint

# Record the supervision the run ACTUALLY performed, measured in this session.
# This is what makes the loss contract auditable after the fact.
manifest['loss_contract_evidence'] = {
    'kind': 'ASSISTANT_ONLY_EXPLICIT_LABEL_MASK',
    'ignore_index': IGNORE_INDEX,
    'train_examples': len(train_examples),
    'dev_examples': len(dev_examples),
    'supervised_tokens_total': SUPERVISED_TOTAL,
    'masked_tokens_total': MASKED_TOTAL,
    'max_assistant_end': MAX_ASSISTANT_END,
    'min_supervised_tokens_per_row': MIN_SUPERVISED,
    'context_length': max_seq_length,
    'truncated_assistant_spans': 0,
    'zero_supervised_rows': 0,
    'governed_chat_template_sha256': GOVERNED_TEMPLATE_SHA256,
    'governed_terminator': GOVERNED_TERMINATOR,
    'role_contract': GOVERNED_ROLE_SEQUENCE,
    'pinned_system_date': GOVERNED_SYSTEM_DATE,
    'train_inference_prompt_parity': 'PROVEN_BY_TOKEN_PREFIX',
    'recipe_hash': PACKAGE['exp002']['recipe_hash'],
}
with open(WORKING / 'manifest.json', 'w', encoding='utf-8') as fh:
    json.dump(manifest, fh, indent=2, sort_keys=True)
print('artifacts saved under', WORKING)
print('loss contract evidence recorded:',
      manifest['loss_contract_evidence']['supervised_tokens_total'], 'supervised tokens')


In [ ]:
# --- Section 14: CHECKSUMS.sha256 - per-file + rollup (Q9) ---
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as fh:
        for chunk in iter(lambda: fh.read(65536), b''):
            h.update(chunk)
    return h.hexdigest()

entries = []
for path in sorted(WORKING.rglob('*')):
    if path.is_file() and path.name != 'CHECKSUMS.sha256':
        rel = path.relative_to(WORKING).as_posix()
        entries.append((rel, sha256_file(path)))
rollup = sha256_text('\n'.join(sorted('%s\t%s' % (rel, digest) for rel, digest in entries)))
lines_out = ['%s  %s' % (digest, rel) for rel, digest in entries]
lines_out.append('# rollup  ' + rollup)
with open(WORKING / 'CHECKSUMS.sha256', 'w', encoding='utf-8') as fh:
    fh.write('\n'.join(lines_out) + '\n')
print('CHECKSUMS.sha256 written; rollup =', rollup)


In [ ]:
# --- Section 15: Optional PRIVATE HF upload via Kaggle Secrets (M2-P0-12) ---
# The token is read by NAME and never printed, never written to any file.
dest = PACKAGE.get('artifact_destination')
hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    secret_name = (dest or {}).get('token_secret_name') or 'HF_TOKEN'
    hf_token = UserSecretsClient().get_secret(secret_name)
except Exception:
    hf_token = None

if dest and dest.get('kind') == 'hf' and dest.get('private') is True and hf_token:
    model.push_to_hub_merged(dest['repo_id'], tokenizer=tokenizer, token=hf_token, save_method='mxfp4')
    print('uploaded adapter to private HF repo:', dest['repo_id'])
else:
    print('no HF destination configured (or secret absent) - local /kaggle/working export is the result')
del hf_token


In [ ]:
# --- Section 16: Finalize - completion marker + logs under /kaggle/working (M2-P0-19, M2-P0-20) ---
env_meta = manifest['environment_metadata']
with open(WORKING / 'training.log', 'a', encoding='utf-8') as fh:
    fh.write('experiment_id=' + PACKAGE['experiment_id'] + '\n')
    fh.write('package_id=' + PACKAGE['package_id'] + '\n')
    fh.write('gpu=' + str(env_meta['gpu']) + '\n')
    fh.write('cuda=' + str(env_meta['cuda']) + '\n')
    fh.write('status=COMPLETED\n')
with open(WORKING / 'COMPLETED', 'w', encoding='utf-8') as fh:
    fh.write(PACKAGE['package_id'] + '\n')
print('run complete - outputs persisted under /kaggle/working')
print('NOTE: run as a committed / Save-Version notebook so /kaggle/working persists.')
